# 1_download_datasets

## Purpose

This notebook handles **data acquisition only** for the P-01 Spain population analysis project. It downloads raw datasets and stores them in a Unity Catalog volume for use by subsequent analysis notebooks.

## What this notebook does:

1. **Creates Unity Catalog infrastructure**:
   * Schema: `geospatial.spain_population_analysis`
   * Volume: `datasets` (for raw data storage)

2. **Downloads source data**:
   * **INE Padrón 2024**: Population counts (provincial and municipal) from Instituto Nacional de Estadística
   * **IGN SIANE**: Administrative boundary polygons (municipios, provincias, CCAA) from Centro Nacional de Información Geográfica

3. **Stores in Unity Catalog volume** (organized by dataset):
   ```
   /Volumes/geospatial/spain_population_analysis/datasets/
     ├── padron/
     │   ├── padron_2024.csv                 (provincial population)
     │   └── padron_municipal_2024.csv       (municipal population)
     │
     └── spain_unidades_administrativas/
         ├── au_AdministrativeUnit_2ndOrder0.gml  (CCAA)
         ├── au_AdministrativeUnit_3rdOrder0.gml  (Provinces)
         └── au_AdministrativeUnit_4thOrder0.gml  (Municipalities)
   ```

## Next Steps

Once data is downloaded and verified here, subsequent notebooks will:
* Convert raw datasets to Delta tables
* Join population to geometries
* Calculate density and aggregate to different admin levels
* Generate H3 spatial indexes
* Create maps and visualizations

---

**Project**: P-01 Phase A - Where does Spain actually live?  
**Notebook**: 1 of N (data acquisition)  
**Last Updated**: May 30, 2026

In [0]:
%run ./0_setup

In [0]:
from pathlib import Path
import pandas as pd

# Unity Catalog volume for storing raw datasets
VOLUME_PATH = Path('/Volumes/geospatial/spain_population_analysis/datasets')

# Organize datasets into separate folders
PADRON_FOLDER = VOLUME_PATH / 'padron'
SIANE_FOLDER = VOLUME_PATH / 'spain_unidades_administrativas'

# Data source URLs
INE_PADRON_PROVINCIAL_URL = "https://www.ine.es/jaxiT3/files/t/es/csv_bdsc/2852.csv?nocab=1"

# For ALL municipalities (not just capitals), we need the complete Padrón Municipal file
# INE provides this as a downloadable CSV file with all ~8,100+ municipalities
# Dataset: Cifras oficiales de población de los municipios españoles
INE_PADRON_MUNICIPAL_URL = "https://www.ine.es/jaxiT3/files/t/csv_bdsc/2881.csv?nocab=1"

# Note: If the above URL doesn't have all municipalities, we'll need to use:
# Manual download from: https://www.ine.es/dyngs/INEbase/es/operacion.htm?c=Estadistica_C&cid=1254736177011&menu=resultados&idp=1254734710990
# Look for "Cifras oficiales de población resultantes de la revisión del Padrón municipal"

IGN_SIANE_URL = "https://www.ign.es/web/resources/docs/IGNCnig/SIANE.zip"

# Output paths in volume (organized by dataset)
PADRON_PROVINCIAL_PATH = PADRON_FOLDER / 'padron_2024.csv'
PADRON_MUNICIPAL_PATH = PADRON_FOLDER / 'padron_municipal_all_2024.csv'  # Updated name to reflect ALL municipalities
SIANE_ZIP_PATH = SIANE_FOLDER / 'SIANE.zip'
SIANE_GPKG_PATH = SIANE_FOLDER / 'SIANE.gpkg'

print("Data Storage Configuration:")
print(f"  Volume: {VOLUME_PATH}")
print(f"\nDataset Folders:")
print(f"  • {PADRON_FOLDER.relative_to(VOLUME_PATH)}")
print(f"  • {SIANE_FOLDER.relative_to(VOLUME_PATH)}")
print(f"\nData Sources:")
print(f"  INE Padrón (Provincial): {INE_PADRON_PROVINCIAL_URL}")
print(f"  INE Padrón (ALL Municipalities): {INE_PADRON_MUNICIPAL_URL}")
print(f"  IGN SIANE: {IGN_SIANE_URL}")
print(f"\nOutput Files:")
print(f"  • padron/{PADRON_PROVINCIAL_PATH.name}")
print(f"  • padron/{PADRON_MUNICIPAL_PATH.name} (ALL municipalities, ~8,100+)")
print(f"  • spain_unidades_administrativas/(GML files)")
print(f"\n⚠️  Note: If automated download doesn't capture all municipalities,")
print(f"   manual download may be required from INE's download center.")
print("\n✓ Ready to download datasets")

In [0]:
%sql
-- Create a dedicated schema for Spain population analysis
CREATE SCHEMA IF NOT EXISTS geospatial.spain_population_analysis
  COMMENT 'P-01 Phase A: Spain population density analysis with H3 spatial indexing';

-- Verify schema was created
DESCRIBE SCHEMA EXTENDED geospatial.spain_population_analysis;

In [0]:
# Download INE Padrón provincial population data to Unity Catalog volume
import urllib.request
import os

print("Downloading INE Padrón provincial population data...")
print(f"Source: {INE_PADRON_PROVINCIAL_URL}")
print(f"Destination: {PADRON_PROVINCIAL_PATH}\n")

try:
    # Ensure directory exists
    PADRON_FOLDER.mkdir(parents=True, exist_ok=True)
    
    # Download
    urllib.request.urlretrieve(INE_PADRON_PROVINCIAL_URL, PADRON_PROVINCIAL_PATH)
    print(f"✓ Successfully downloaded")
    print(f"  File size: {os.path.getsize(PADRON_PROVINCIAL_PATH) / 1024:.2f} KB")
    
    # Quick preview
    print("\nData preview (first 3 rows):")
    df_preview = pd.read_csv(PADRON_PROVINCIAL_PATH, nrows=3, encoding='utf-8', sep=';')
    display(df_preview)
    
    print(f"\n✓ Provincial data stored in: {PADRON_PROVINCIAL_PATH}")
except Exception as e:
    print(f"✗ Download failed: {e}")

In [0]:
# Download IGN SIANE administrative boundaries to Unity Catalog volume
import urllib.request
import zipfile
import os

print("Downloading IGN SIANE administrative boundaries...")
print(f"Source: {IGN_SIANE_URL}")
print("Note: Large file (~100+ MB), may take a moment...\n")

try:
    # Ensure directory exists
    SIANE_FOLDER.mkdir(parents=True, exist_ok=True)
    
    # Download ZIP
    print("[1/3] Downloading ZIP archive...")
    urllib.request.urlretrieve(IGN_SIANE_URL, SIANE_ZIP_PATH)
    print(f"  ✓ Downloaded: {os.path.getsize(SIANE_ZIP_PATH) / (1024*1024):.2f} MB")
    
    # Extract GML files
    print("\n[2/3] Extracting GML files...")
    with zipfile.ZipFile(SIANE_ZIP_PATH, 'r') as zip_ref:
        # Extract all GML files to the SIANE folder
        gml_files = [f for f in zip_ref.namelist() if f.endswith('.gml')]
        
        if gml_files:
            print(f"  Found {len(gml_files)} GML files")
            for gml_file in gml_files:
                zip_ref.extract(gml_file, SIANE_FOLDER)
            print(f"  ✓ Extracted to: {SIANE_FOLDER}")
        else:
            print("  ✗ No GML files found in archive")
    
    # Cleanup ZIP
    print("\n[3/3] Cleaning up...")
    os.remove(SIANE_ZIP_PATH)
    print("  ✓ Removed temporary ZIP file")
    
    print(f"\n✓ SIANE GML files stored in: {SIANE_FOLDER}")
    
except Exception as e:
    print(f"\n✗ Download/extraction failed: {e}")
    print("\nNote: SIANE data is already manually uploaded to the volume.")

In [0]:
# Download ALL municipal-level population data from INE
# ⚠️ INE's jaxiT3 API is limited to single provinces - manual download required
import urllib.request
import os

print("="*60)
print("INE MUNICIPAL DATA - MANUAL DOWNLOAD REQUIRED")
print("="*60)
print("\n⚠️  The INE jaxiT3 CSV API only provides data for ONE province at a time.")
print("   Dataset 2881 = Madrid only (180 municipalities)")
print("   To get all ~8,100+ municipalities, manual download is required.\n")

print("MANUAL DOWNLOAD STEPS:")
print("-" * 60)
print("\n1. Visit INE Download Portal:")
print("   https://www.ine.es/dynt3/inebase/index.htm?padre=517\n")

print("2. Look for one of these datasets:")
print("   • 'Cifras oficiales de población de los municipios españoles'")
print("   • 'Población por municipios y sexo'")
print("   • Look for the most recent year with ALL municipalities\n")

print("3. Download the COMPLETE CSV file:")
print("   • File should be 5-20 MB (large file = all municipalities)")
print("   • Format: CSV with semicolon separator")
print("   • Columns should include: Municipio code, Name, Sex, Year, Population\n")

print("4. Upload the CSV to this location:")
print(f"   {PADRON_MUNICIPAL_PATH}\n")

print("5. Verify after upload (run this):")
print("   df = pd.read_csv(PADRON_MUNICIPAL_PATH, sep=';')")
print("   unique_munis = df.iloc[:,0].str[:5].nunique()")
print("   print(f'Municipalities: {unique_munis}')  # Should be ~8,100+\n")

print("=" * 60)
print("\nℹ️  Alternative: Use provincial data only (52 capitals)")
print("   If complete municipal data is not critical, the existing")
print("   padron_municipal_2024.csv has 52 provincial capitals.\n")

print("ℹ️  Expected Coverage:")
print("   • 52 provinces")
print("   • ~8,131 municipalities (varies by year)")
print("   • Population by sex and year (1996-2024)\n")

# Check if file already exists
if PADRON_MUNICIPAL_PATH.exists():
    print("✓ File already exists - verifying coverage...\n")
    try:
        df_check = pd.read_csv(PADRON_MUNICIPAL_PATH, sep=';', encoding='utf-8')
        first_col = df_check.columns[0]
        muni_count = df_check[first_col].astype(str).str[:5].nunique()
        
        if muni_count > 7000:
            print(f"✓ COMPLETE: {muni_count:,} municipalities detected!")
            print("  No action needed - data is already complete.\n")
        else:
            print(f"⚠️  INCOMPLETE: Only {muni_count:,} municipalities detected.")
            print("  Please follow the manual download steps above.\n")
    except Exception as e:
        print(f"✗ Error reading existing file: {e}\n")
else:
    print("✗ File not found - please follow manual download steps above.\n")

## ⚠️ REQUIRED: Manual Download for ALL Municipalities

**Why manual download?** INE's jaxiT3 API only provides data for individual provinces. To get all ~8,100+ municipalities across all of Spain, you must download the complete datasets from INE's portal.

---

### Step-by-Step Download Guide

#### 1. Go to INE Municipal Population Data Page

**Visit**: https://www.ine.es/dynt3/inebase/index.htm?padre=6232&capsel=6233

This page contains all the municipal population datasets organized by province and national aggregates.

---

#### 2. Download ALL Datasets from "00 Nacional" Section

**Important**: You need to download **all datasets** listed under the **"00 Nacional"** section. This section contains the complete national data with all municipalities.

**For each dataset in the "00 Nacional" section**:

1. Click on the dataset link
2. Click the **download/export button** (usually a download icon or "Descargar" button)
3. **Select CSV format**
4. **Separator**: Choose **semicolon (`;`)** as the separator
5. **Encoding**: UTF-8 (default)
6. Save the file to your computer

**Repeat this process** for each dataset in the "00 Nacional" section until you have downloaded all of them.

---

#### 3. Upload to Databricks

Once you have downloaded all the CSV files from the "00 Nacional" section:

**Upload all files** to the Databricks volume folder:
* **Target folder**: `/Volumes/geospatial/spain_population_analysis/datasets/padron/`
* Upload all CSV files directly to this folder
* Keep the original filenames from INE

**Upload methods**:
* Drag and drop files into the Databricks UI
* Use the Databricks CLI
* Use the "Upload to Volume" option in the Data Explorer

---

### Summary

✓ Go to: https://www.ine.es/dynt3/inebase/index.htm?padre=6232&capsel=6233  
✓ Download **all datasets** in the **"00 Nacional"** section  
✓ Format: **CSV** with **semicolon (`;`)** separator  
✓ Upload all files to: `/Volumes/geospatial/spain_population_analysis/datasets/padron/`  
✓ Expected coverage: ~8,100+ municipalities across all 52 Spanish provinces

In [0]:
%sql
-- Create a managed volume to store raw datasets
CREATE VOLUME IF NOT EXISTS geospatial.spain_population_analysis.datasets;

In [0]:
# Verify what files are in the volume (organized by dataset folders)
import os

def list_files_recursive(folder, indent=2):
    """Recursively list files in folder with sizes"""
    if not folder.exists():
        print(f"{' '*indent}✗ Folder not found")
        return
    
    for item in sorted(folder.iterdir()):
        if item.is_dir():
            print(f"{' '*indent}📁 {item.name}/")
            list_files_recursive(item, indent + 2)
        else:
            size_mb = os.path.getsize(item) / (1024 * 1024)
            print(f"{' '*indent}✓ {item.name:40s} ({size_mb:8.2f} MB)")

print("Files in volume (organized by dataset):")
print(f"  Location: {VOLUME_PATH}\n")

if VOLUME_PATH.exists():
    list_files_recursive(VOLUME_PATH)
    
    print("\n" + "="*60)
    print("Download Status Summary:")
    print(f"\n  Padrón Folder ({PADRON_FOLDER.name}):")
    print(f"    {'✓' if PADRON_PROVINCIAL_PATH.exists() else '✗'} Provincial data: {PADRON_PROVINCIAL_PATH.exists()}")
    
    # Check municipal data with municipality count
    if PADRON_MUNICIPAL_PATH.exists():
        df_check = pd.read_csv(PADRON_MUNICIPAL_PATH, sep=';', encoding='utf-8')
        first_col = df_check.columns[0]
        muni_count = df_check[first_col].astype(str).str[:5].nunique()
        
        status = "✓" if muni_count > 7000 else "⚠️"
        print(f"    {status} Municipal data: {PADRON_MUNICIPAL_PATH.exists()} ({muni_count:,} municipalities)")
        
        if muni_count < 7000:
            print(f"\n    ⚠️  WARNING: Only {muni_count:,} municipalities detected!")
            print(f"       Expected: ~8,100+ municipalities")
            print(f"       → Manual download may be required (see cell above)")
    else:
        print(f"    ✗ Municipal data: {PADRON_MUNICIPAL_PATH.exists()}")
    
    print(f"\n  SIANE Folder ({SIANE_FOLDER.name}):")
    if SIANE_FOLDER.exists():
        gml_count = len(list(SIANE_FOLDER.glob('*.gml')))
        print(f"    ✓ GML files: {gml_count}")
    else:
        print(f"    ✗ Folder not found")
    
    print("\n" + "="*60)
    print("\n✓ Verification complete")
else:
    print("  ✗ Volume directory not found")

## Alternative: Download SIANE Manually

The automated SIANE download URL appears to be unavailable (404 error). 

**Manual download options:**

1. **IGN Centro de Descargas**: https://centrodedescargas.cnig.es/CentroDescargas/
   * Navigate to: "Información Geográfica de Referencia" → "Unidades administrativas"
   * Download the GeoPackage for all of Spain

2. **Alternative source - Natural Earth**:
   * For a quick test, you can use Natural Earth administrative boundaries
   * Spain admin level 1 (CCAA) and level 2 (provinces) available

3. **Upload manually**:
   * Download the SIANE.gpkg file
   * Upload it to `/Volumes/geospatial/spain_population_analysis/datasets/SIANE.gpkg`

Once uploaded, the analysis notebooks will be able to read it from the volume.